# Choza mid-resolution pilot

Cadences 000–002: divide each time row by a running median of width 101 (reflected edges), then Seticore at S/N 5, 7, 10 and requested maximum drift 40 Hz/s. Find-event level 3 excludes zero drift; no additional SNR or drift cut is applied. Originals remain unchanged. `raw_hit_counts.csv` reports actual drift extrema and counts beyond ±40 Hz/s.

Run the script in a terminal for a persistent job. Do not start a second copy while the pilot is running.


In [ ]:
from pathlib import Path
import sys, subprocess
import pandas as pd
ROOT = Path('/home/wlll2x/midres_narrow/carmen_midres')
SCRIPT = ROOT / 'scripts/choza_midres_search.py'
OUT = ROOT / 'outputs/choza_pilot'


In [ ]:
subprocess.run([sys.executable, str(SCRIPT), '--dry-run'], check=True)


## Launch from a terminal
```bash
cd /home/wlll2x/midres_narrow/carmen_midres
nohup python -u scripts/choza_midres_search.py > outputs/choza_pilot/run.log 2>&1 &
tail -f outputs/choza_pilot/run.log
```
Successful individual searches are reused when rerunning with identical settings. Use a new `--output` directory for different settings.


In [ ]:
for name in ['raw_hit_counts.csv', 'event_summary.csv']:
    path = OUT / name
    if path.exists():
        print(name)
        display(pd.read_csv(path))
    else:
        print(name, 'not yet available')


In [ ]:
tables = []
for path in sorted(OUT.glob('cadence_*/snr_*/events.csv')):
    table = pd.read_csv(path)
    table['cadence'] = path.parent.parent.name
    table['threshold'] = path.parent.name
    tables.append(table)
events = pd.concat(tables, ignore_index=True) if tables else pd.DataFrame()
display(events)


## Candidate waterfall
Select a surviving event row. This reads a narrow frequency window from each original scan and plots the ABACAD cadence. Inspect the candidate in all six scans before interpreting it. Fine-resolution comparison is a subsequent step using matching observation files/results.


In [ ]:
import json
import numpy as np
import h5py
import matplotlib.pyplot as plt

def plot_candidate(event, half_width_mhz=0.15):
    settings = json.loads((OUT / 'settings.json').read_text())
    scans = settings['manifest'][event['cadence']]
    fig, axes = plt.subplots(6, 1, figsize=(11, 13), constrained_layout=True)
    for i, (ax, scan) in enumerate(zip(axes, scans)):
        with h5py.File(scan['path']) as f:
            d = f['data']
            a = d.attrs
            center = float(event['Freq']) + float(event['DriftRate']) * (float(a['tstart']) - float(event['MJD'])) * 86400 / 1e6
            freq = float(a['fch1']) + np.arange(d.shape[-1]) * float(a['foff'])
            indices = np.flatnonzero(abs(freq-center) <= half_width_mhz)
            if not len(indices):
                ax.set_title('Frequency outside this scan'); continue
            lo, hi = indices[0], indices[-1]+1
            data = d[:, 0, lo:hi]
            data = data / np.maximum(np.median(data, axis=1, keepdims=True), np.finfo(float).tiny)
            ax.imshow(data, aspect='auto', origin='lower', extent=[freq[lo], freq[hi-1], 0, d.shape[0]*float(a['tsamp'])], vmin=np.percentile(data, 5), vmax=np.percentile(data, 99))
            ax.set_title(f"{'ON' if i%2==0 else 'OFF'}: {scan['source']}")
            ax.set_ylabel('Time (s)')
    axes[-1].set_xlabel('Frequency (MHz)')
    return fig

if not events.empty:
    plot_candidate(events.iloc[0])
else:
    print('No surviving candidates available yet.')
